# 05 · Chia bằng chứng và tạo chỉ mục

**Input:** Output 04. **Thiết bị:** CPU hoặc GPU. **Output:** 05_index.

Gom lời giảng cùng interval thành chunk, giữ OCR và ảnh làm bằng chứng.
Mặc định chỉ thêm mô tả ảnh/vùng nội dung đã được con người xác minh.
Đếm token bằng tokenizer E5, tính cả tiêu đề và prefix để không cắt ngầm.
Lưu vectors chuẩn hóa cùng SQLite FTS5, model revision và thứ tự chunk IDs.
Output kèm dữ liệu nguồn để bước 06 chỉ cần đính kèm một notebook.


## 1. Môi trường

Bật **Internet** để cài thư viện/tải model. Chạy mỗi notebook trong phiên Kaggle mới, với Input và GPU như ghi ở trên.


In [ ]:
%pip install -q "sentence-transformers>=3.4,<6" "transformers>=4.51.3,<5"


## 2. Hàm xử lý

Các hàm nhúng trực tiếp; không cần tải file Python đi kèm. Mở ô thu gọn để xem hoặc chỉnh thuật toán.


In [ ]:
"""Shared artifact contract. Embedded into every notebook by build_notebooks.py."""

from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import re
import shutil
import subprocess
import threading
import unicodedata
from datetime import datetime, timezone

# transformers' from_pretrained builds models under a process-wide "meta device" context
# that is not thread-safe: two loads at once (OCR thread and main thread) fail with
# "Cannot copy out of meta tensor". Every model is loaded once per session, one at a time.
MODEL_LOAD_LOCK = threading.RLock()
_SHARED_MODELS = {}


def shared_model(key, loader):
    """Load a model once per session (reused by every video) and never two at once."""
    with MODEL_LOAD_LOCK:
        if key not in _SHARED_MODELS:
            _SHARED_MODELS[key] = loader()
        return _SHARED_MODELS[key]


SCHEMA_VERSION = "lecture-rag-v1"
PIPELINE_VERSION = "l25-v1"
STAGE_NAMES = {
    1: "01_slides", 2: "02_visual", 3: "03_audio",
    4: "04_bundle", 5: "05_index", 6: "06_retrieval",
}


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def fingerprint(value):
    data = json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False)
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def relative_asset(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if not path.is_relative_to(root):
        raise ValueError(f"Asset nằm ngoài thư mục kết quả: {name}")
    return path


def run_command(arguments):
    process = subprocess.run(
        [str(value) for value in arguments], capture_output=True, text=True,
        encoding="utf-8", errors="replace",
    )
    if process.returncode:
        raise RuntimeError(
            f"{arguments[0]} thất bại ({process.returncode}):\n{process.stderr[-3000:]}"
        )
    return process.stdout


def package_versions(names):
    versions = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def find_video(override=None, input_root="/kaggle/input"):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy video: {path}")
        return path
    candidates = sorted(
        path for path in Path(input_root).rglob("*")
        if path.is_file()
        and path.suffix.lower() in {".mp4", ".mkv", ".mov", ".webm"}
        and path.stem.lower() in {"l25_v001", "tl25_v001"}
    )
    if len(candidates) != 1:
        raise ValueError(
            "Cần đúng một video L25_V001/TL25_V001. Add dataset "
            "thnhtrungnguynmif/l25a-lecture-videos-media hoặc đặt VIDEO_PATH. "
            f"Tìm thấy: {[str(path) for path in candidates]}"
        )
    return candidates[0]


def probe_video(path):
    if not shutil.which("ffprobe") or not shutil.which("ffmpeg"):
        raise RuntimeError("Cần FFmpeg và FFprobe trong PATH.")
    data = json.loads(run_command([
        "ffprobe", "-v", "error", "-show_format", "-show_streams",
        "-of", "json", str(path),
    ]))
    videos = [stream for stream in data["streams"] if stream["codec_type"] == "video"]
    audios = [stream for stream in data["streams"] if stream["codec_type"] == "audio"]
    if not videos:
        raise ValueError("Tệp không có luồng video.")
    info = data.get("format", {})
    origin = float(info.get("start_time", 0) or 0)
    duration = float(info.get("duration") or videos[0].get("duration") or 0)
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Không đọc được thời lượng video hợp lệ.")
    return {
        "filename": Path(path).name,
        "sha256": file_sha256(path),
        "duration_sec": duration,
        "timeline_origin_sec": origin,
        "video_start_sec": float(videos[0].get("start_time", origin)),
        "audio_start_sec": float(audios[0].get("start_time", origin)) if audios else None,
        "has_audio": bool(audios),
        "width": int(videos[0]["width"]),
        "height": int(videos[0]["height"]),
        "clock": "original_presentation_timestamp_minus_format_start_time",
    }


def begin_stage(output_root, stage):
    directory = Path(output_root) / STAGE_NAMES[stage]
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "_RUNNING.json", {
        "stage": stage, "started_utc": datetime.now(timezone.utc).isoformat(),
    })
    return directory


def finish_stage(directory, stage, video_id, source, parameters, parents=(), extra=None):
    directory = Path(directory)
    files = {
        path.relative_to(directory).as_posix(): file_sha256(path)
        for path in sorted(directory.rglob("*"))
        if path.is_file()
        and path.name != "manifest.json"
        and not any(part.startswith("_") for part in path.relative_to(directory).parts)
        and path.suffix != ".tmp"
    }
    payload = {
        "schema_version": SCHEMA_VERSION,
        "pipeline_version": PIPELINE_VERSION,
        "stage": stage,
        "video_id": video_id,
        "source": source,
        "parameters": parameters,
        "parents": {str(parent["stage"]): parent["artifact_id"] for parent in parents},
        "files": files,
        **(extra or {}),
    }
    payload["artifact_id"] = fingerprint(payload)
    payload["created_utc"] = datetime.now(timezone.utc).isoformat()
    write_json(directory / "manifest.json", payload)
    (directory / "_RUNNING.json").unlink(missing_ok=True)
    print(f"Hoàn tất {STAGE_NAMES[stage]}: {directory}")
    return payload


def load_stage(directory, stage=None, video_id="L25_V001", verify=True):
    directory = Path(directory)
    if (directory / "_RUNNING.json").exists():
        raise ValueError(f"Bước xử lý chưa hoàn tất; cần chạy lại: {directory}")
    manifest = read_json(directory / "manifest.json")
    if manifest.get("schema_version") != SCHEMA_VERSION:
        raise ValueError(f"Schema không tương thích: {directory}")
    if stage is not None and manifest.get("stage") != stage:
        raise ValueError(f"Cần stage {stage}, nhận {manifest.get('stage')}: {directory}")
    if manifest.get("video_id") != video_id:
        raise ValueError(f"Sai video_id: {manifest.get('video_id')} != {video_id}")
    if manifest.get("pipeline_version") != PIPELINE_VERSION:
        raise ValueError(f"Sai pipeline_version: {directory}")
    signed = {key: value for key, value in manifest.items()
              if key not in {"artifact_id", "created_utc"}}
    if fingerprint(signed) != manifest["artifact_id"]:
        raise ValueError(f"Manifest đã thay đổi sau khi xuất: {directory}")
    if verify:
        for name, expected in manifest["files"].items():
            path = relative_asset(directory, name)
            if not path.is_file() or file_sha256(path) != expected:
                raise ValueError(f"Asset bị thiếu hoặc không khớp checksum: {path}")
    return manifest


def find_stage(stage, override=None, video_id="L25_V001",
               search_roots=("/kaggle/input", "/kaggle/working")):
    if override:
        directory = Path(override)
        return directory, load_stage(directory, stage, video_id)
    matches = []
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for path in root.rglob("manifest.json"):
            try:
                manifest = read_json(path)
            except (ValueError, OSError):
                continue
            if (manifest.get("schema_version") == SCHEMA_VERSION
                    and manifest.get("stage") == stage
                    and manifest.get("video_id") == video_id
                    and manifest.get("pipeline_version") == PIPELINE_VERSION):
                matches.append((path.parent, manifest))
    identities = {manifest.get("artifact_id") for _, manifest in matches}
    if not matches or len(identities) != 1:
        raise ValueError(
            f"Cần một phiên bản rõ ràng của {STAGE_NAMES[stage]}. "
            "Add Input → Notebook Output của bước trước, hoặc đặt STAGE_XX_DIR. "
            f"Các thư mục tìm được: {[str(path) for path, _ in matches]}"
        )
    directory = sorted(matches, key=lambda item: str(item[0]))[0][0]
    return directory, load_stage(directory, stage, video_id)


def require_same_source(*manifests):
    identities = {
        (item["video_id"], item["source"]["sha256"], item["pipeline_version"])
        for item in manifests
    }
    if len(identities) != 1:
        raise ValueError("Các nhánh không được tạo từ cùng video/pipeline.")


def require_parent(child, parent):
    require_same_source(child, parent)
    if child["parents"].get(str(parent["stage"])) != parent["artifact_id"]:
        raise ValueError(
            f"Stage {child['stage']} dùng phiên bản stage {parent['stage']} khác. "
            "Chạy lại nhánh với cùng kết quả phát hiện slide."
        )


def verify_intervals(intervals, duration, tolerance=0.03):
    if not intervals:
        raise ValueError("Danh sách interval rỗng.")
    ids = set()
    previous_end = 0.0
    for index, item in enumerate(intervals):
        start, end = float(item["start_sec"]), float(item["end_sec"])
        if not (math.isfinite(start) and math.isfinite(end)
                and 0 <= start < end <= duration + tolerance):
            raise ValueError(f"Interval không hợp lệ: {item}")
        if abs(start - previous_end) > tolerance:
            raise ValueError(f"Khoảng trống/chồng lấn ở interval {index}.")
        if item["segment_id"] in ids:
            raise ValueError("Trùng segment_id.")
        ids.add(item["segment_id"])
        previous_end = end
    if abs(previous_end - duration) > tolerance:
        raise ValueError("Interval cuối không bao phủ hết video.")


def owner_interval(intervals, time_sec):
    # Half-open intervals: a word exactly at a boundary belongs to the next one.
    for item in intervals:
        if item["start_sec"] <= time_sec < item["end_sec"]:
            return item
    if intervals and math.isclose(time_sec, intervals[-1]["end_sec"], abs_tol=1e-6):
        return intervals[-1]
    raise ValueError(f"Timestamp nằm ngoài lecture: {time_sec}")


def copy_stage_assets(source, destination, manifest):
    for name in manifest["files"]:
        target = relative_asset(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(relative_asset(source, name), target)


def normalize_search(text):
    text = unicodedata.normalize("NFD", text.casefold().replace("đ", "d"))
    return "".join(char for char in text if not unicodedata.combining(char))


def timestamp_label(seconds):
    seconds = max(0, int(seconds))
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}" if hours else f"{minutes:02}:{seconds:02}"


def unique_records(records, key):
    result = {}
    for record in records:
        if record[key] in result:
            raise ValueError(f"Trùng {key}: {record[key]}")
        result[record[key]] = record
    return result


def read_pcm_audio(path, sample_rate=16000):
    import numpy as np
    import wave

    with wave.open(str(path), "rb") as stream:
        if (stream.getframerate() != sample_rate or stream.getnchannels() != 1
                or stream.getsampwidth() != 2):
            raise ValueError("Audio phải là PCM signed 16-bit, mono, 16 kHz.")
        return np.frombuffer(stream.readframes(stream.getnframes()), dtype="<i2").copy()


def save_pcm_clip(path, pcm, start_sample, end_sample, sample_rate=16000):
    import wave

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as stream:
        stream.setnchannels(1)
        stream.setsampwidth(2)
        stream.setframerate(sample_rate)
        stream.writeframes(pcm[start_sample:end_sample].astype("<i2").tobytes())


In [ ]:
"""Evidence chunks, multilingual E5 vectors, and a Vietnamese-friendly FTS5 index."""


def split_text_tokens(text, tokenizer, limit=350, overlap=40):
    if not 0 <= overlap < limit:
        raise ValueError("Cần 0 <= overlap < giới hạn token.")
    tokens = tokenizer.encode(text, add_special_tokens=False)
    parts = []
    start = 0
    while start < len(tokens):
        end = min(len(tokens), start + limit)
        part = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if part:
            parts.append(part)
        if end == len(tokens):
            break
        start = end - overlap
    return parts


def split_layout_tokens(text, tokenizer, limit, overlap):
    """Keep literal OCR whitespace; overlap complete lines rather than decoded code tokens."""
    parts, pending = [], []
    for line in text.splitlines():
        proposed = "\n".join([*pending, line])
        if pending and len(tokenizer.encode(proposed, add_special_tokens=False)) > limit:
            parts.append("\n".join(pending))
            tail = []
            for previous in reversed(pending):
                if len(tokenizer.encode("\n".join([previous, *tail]), add_special_tokens=False)) > overlap:
                    break
                tail.insert(0, previous)
            pending = tail
            if len(tokenizer.encode("\n".join([*pending, line]), add_special_tokens=False)) > limit:
                pending = []
        if len(tokenizer.encode(line, add_special_tokens=False)) > limit:
            parts.extend(split_text_tokens(line, tokenizer, limit, overlap))
        else:
            pending.append(line)
    if pending:
        parts.append("\n".join(pending))
    return [part for part in parts if part.strip()]


def usable_figure_description(text):
    """Generated no-figure placeholders, punctuation and repetition are not evidence."""
    text = (text or "").strip()
    folded = unicodedata.normalize("NFD", text.lower())
    folded = "".join(c for c in folded if unicodedata.category(c) != "Mn").replace("đ", "d")
    return bool(re.search(r"[^\W\d_]", text) and not re.match(
        r"^(?:khong co (?:hinh|bieu do|so do)|no (?:figure|image|diagram)|there is no (?:figure|image|diagram))\b", folded)
        and not re.search(r"(\b\w+(?:\s+\w+){0,7}\b)(?:\s+\1){7,}", folded))


def group_speech_spans(spans, tokenizer, maximum_tokens=180, maximum_seconds=50.0):
    """Keep complete sentences together; never merge across visual intervals."""
    groups, pending = [], []

    def finish():
        if pending:
            groups.append({
                "source_type": "speech", "text": " ".join(span["text"] for span in pending),
                "start_sec": min(span["start_sec"] for span in pending),
                "end_sec": max(span["end_sec"] for span in pending),
                "source_ids": [span["span_id"] for span in pending],
                "timing_precision": "word_span",
            })

    for span in spans:
        proposed = " ".join(item["text"] for item in [*pending, span])
        if pending and (len(tokenizer.encode(proposed, add_special_tokens=False)) > maximum_tokens
                        or span["end_sec"] - pending[0]["start_sec"] > maximum_seconds
                        or span["start_sec"] - pending[-1]["end_sec"] > 3.0):
            finish()
            pending = []
        pending.append(span)
    finish()
    return groups


def make_evidence_chunks(segments, fallback_spans, tokenizer, *,
                         chunk_tokens=350, overlap_tokens=40, model_limit=512,
                         video_id="L25_V001", speech_chunk_tokens=180,
                         speech_maximum_seconds=50.0, include_generated_descriptions=False,
                         dedupe_previous=False, passage_prefix="passage: ", generated_semantic_only=False):
    """passage_prefix: "passage: " for E5, "" for bge-m3. dedupe_previous: index only the OCR lines not already on the previous slide (stage 04
    ocr_text_new), so a repeated slide does not crowd the search with copies."""
    if not 0 <= overlap_tokens < chunk_tokens <= model_limit - 32:
        raise ValueError("Giới hạn chunk không tương thích model tokenizer.")
    chunks = []
    seen_coarse = set()
    coarse = unique_records(fallback_spans, "span_id")
    for segment in segments:
        if segment.get("ad_flagged") or (segment.get("content_kind") == "ad" and not segment.get("ad_suspect")):
            # Ads are not lesson evidence; their speech is kept in stage 04 only. An ad_suspect
            # segment (its speech sounds like the lesson, stage 04) is indexed by its speech.
            # ad_flagged (ad filter in "flag" mode): OCR and speech are kept in stage 04 only.
            seen_coarse.update(segment["speech"]["fallback_span_ids"])
            continue
        visual = segment["visual"]
        visual_failed = visual.get("ocr_quality", {}).get("evidence_excluded", False)
        visual_start = visual.get("representative_content_onset_sec", segment["start_sec"])
        if not segment["start_sec"] <= visual_start < segment["end_sec"]:
            raise ValueError("Representative visual onset must be inside its interval")
        visual_timing = ("visual_state" if "representative_content_onset_sec" in visual else "visual_interval")
        # Bound the title separately so it cannot consume the encoder budget.
        title_tokens = tokenizer.encode("" if visual_failed else visual.get("title_clean", visual.get("title", "")),
                                        add_special_tokens=False)[:64]
        title = tokenizer.decode(title_tokens, skip_special_tokens=True).strip()
        header = f"{passage_prefix}{title}\n" if title else passage_prefix
        budget = min(chunk_tokens, model_limit
                     - len(tokenizer.encode(header, add_special_tokens=True)) - 8)
        units = group_speech_spans(
            segment["speech"]["spans"], tokenizer,
            min(speech_chunk_tokens, budget), speech_maximum_seconds,
        )
        for span_id in segment["speech"]["fallback_span_ids"]:
            if span_id in seen_coarse:
                continue
            seen_coarse.add(span_id)
            span = coarse[span_id]
            units.append({
                "source_type": "speech", "text": span["text"],
                "start_sec": span["start_sec"], "end_sec": span["end_sec"],
                "source_ids": [span_id], "timing_precision": "asr_segment",
                "uncertainty": span.get("uncertainty"),
            })
        for source_type, text in [
            ("ocr", visual["ocr_text_new"] if dedupe_previous and "ocr_text_new" in visual
             else visual.get("ocr_text_clean", visual.get("ocr_text", ""))),
            ("visual", visual.get("visual_description", "")),
        ]:
            if visual_failed:
                continue
            # Older stage-02 bundles may store malformed JSON or degenerate
            # generation (e.g. repeated punctuation) as visual_description.
            # Preserve those raw records in lecture/, but never index them.
            if source_type == "visual" and visual.get("description_status") != "json":
                continue
            if (source_type == "visual" and not include_generated_descriptions
                    and visual.get("description_verification") != "human_verified"):
                continue
            generated = source_type == "visual" and visual.get("description_verification") != "human_verified"
            if generated and not usable_figure_description(text):
                continue
            if text.strip():
                units.append({
                    "source_type": source_type, "text": text,
                    "start_sec": visual_start, "end_sec": segment["end_sec"],
                    "source_ids": [visual["occurrence_id"]],
                    "timing_precision": visual_timing,
                    "uncertainty": "OCR/mô tả tự động; đối chiếu hình gốc khi cần.",
                    **({"layout_preserved": True} if source_type == "ocr" and visual.get("ocr_layout_preserved") else {}),
                    **({"verification": "model_generated_unverified",
                        "retrieval_policy": "semantic_only" if generated_semantic_only else "hybrid"} if generated else {}),
                })
        for region in visual.get("regions", []):
            if visual_failed or region.get("noise"):
                continue
            if (not include_generated_descriptions and region.get("extraction_status")
                    not in {"human_verified", "hybrid_ocr_unverified"}):
                continue
            text = "\n".join(value for value in
                             [region.get("text"), region.get("description")] if value)
            generated = region.get("extraction_status") == "model_generated_unverified"
            if generated and not usable_figure_description(text):
                continue
            if text.strip():
                units.append({
                    "source_type": "region", "text": text,
                    "start_sec": visual_start, "end_sec": segment["end_sec"],
                    "source_ids": [region["region_id"]],
                    "image_path": region["image_path"],
                    "timing_precision": visual_timing,
                    "uncertainty": "Vùng và nội dung do mô hình đề xuất, chưa xác minh.",
                    **({"verification": "model_generated_unverified",
                        "retrieval_policy": "semantic_only" if generated_semantic_only else "hybrid"} if generated else {}),
                })
        for unit_index, unit in enumerate(units):
            split = split_layout_tokens if unit.get("layout_preserved") else split_text_tokens
            parts = split(unit["text"], tokenizer, budget, min(overlap_tokens, budget - 1))
            for part_index, text in enumerate(parts):
                embedding_text = header + text
                count = len(tokenizer.encode(embedding_text, add_special_tokens=True))
                if count > model_limit:
                    raise ValueError("Chunk vượt giới hạn encoder; giảm CHUNK_TOKENS.")
                chunks.append({
                    **{key: value for key, value in unit.items() if key != "text"},
                    "chunk_id": f"{segment['segment_id']}:u{unit_index:04d}:p{part_index:03d}",
                    "video_id": video_id, "segment_id": segment["segment_id"],
                    "sequence": segment["sequence"], "text": text,
                    "title": title, "embedding_text": embedding_text,
                    "encoded_tokens": count,
                    "image_path": unit.get("image_path", visual.get("content_image_path")
                                           or visual["image_path"]),
                    "keyframe_time_sec": visual["keyframe_time_sec"],
                    "occurrence_id": visual["occurrence_id"],
                })
    if not chunks:
        raise ValueError("Không có bằng chứng để lập chỉ mục; kiểm tra OCR/ASR.")
    if len({chunk["chunk_id"] for chunk in chunks}) != len(chunks):
        raise ValueError("Trùng chunk_id.")
    if seen_coarse != set(coarse):
        raise ValueError("Có fallback span chưa được đưa vào chỉ mục.")
    return chunks


def build_lexical_index(path, chunks):
    from contextlib import closing
    import sqlite3

    with closing(sqlite3.connect(path)) as connection:
        connection.execute("DROP TABLE IF EXISTS evidence")
        try:
            connection.execute(
                "CREATE VIRTUAL TABLE evidence USING fts5("
                "chunk_id UNINDEXED, text, tokenize='unicode61 remove_diacritics 2')"
            )
        except sqlite3.OperationalError as error:
            raise RuntimeError("Python/SQLite này không hỗ trợ FTS5.") from error
        connection.executemany(
            "INSERT INTO evidence(chunk_id, text) VALUES (?, ?)",
            [(chunk["chunk_id"], normalize_search(
                f"{chunk['title']}\n{chunk['text']}"
            )) for chunk in chunks if chunk.get("retrieval_policy") != "semantic_only"],
        )
        connection.commit()


def run_indexing(stage04_dir, output_root, video_id="L25_V001", *,
                 embedding_model="intfloat/multilingual-e5-small", revision="main",
                 chunk_tokens=350, overlap_tokens=40, batch_size=32, device=None,
                 speech_chunk_tokens=180, speech_maximum_seconds=50.0,
                 include_generated_descriptions=False, dedupe_previous=False, generated_semantic_only=False):
    import numpy as np
    # E5 was trained with "query: " / "passage: " prefixes; bge-m3 uses none.
    query_prefix, passage_prefix = ("query: ", "passage: ") if "e5" in embedding_model.lower() else ("", "")
    from sentence_transformers import SentenceTransformer

    parent = load_stage(stage04_dir, 4, video_id)
    stage04_dir = Path(stage04_dir)
    segments = read_json(stage04_dir / "segments.json")
    fallback_spans = read_json(stage04_dir / "fallback_spans.json")
    directory = begin_stage(output_root, 5)
    model = shared_model(("e5", embedding_model, revision, device),
                         lambda: SentenceTransformer(embedding_model, revision=revision, device=device))
    model_limit = min(int(model.max_seq_length), 512)
    chunks = make_evidence_chunks(
        segments, fallback_spans, model.tokenizer, chunk_tokens=chunk_tokens,
        overlap_tokens=overlap_tokens, model_limit=model_limit, video_id=video_id,
        speech_chunk_tokens=speech_chunk_tokens, speech_maximum_seconds=speech_maximum_seconds,
        include_generated_descriptions=include_generated_descriptions, dedupe_previous=dedupe_previous,
        passage_prefix=passage_prefix,
        generated_semantic_only=generated_semantic_only,
    )
    ad_review = []
    if any("spans_before_ad_filter" in s["speech"] for s in segments):
        before = [{**s, "speech": {**s["speech"], "spans": s["speech"].get("spans_before_ad_filter", s["speech"]["spans"])}}
                  for s in segments]
        original = make_evidence_chunks(
            before, fallback_spans, model.tokenizer, chunk_tokens=chunk_tokens,
            overlap_tokens=overlap_tokens, model_limit=model_limit, video_id=video_id,
            speech_chunk_tokens=speech_chunk_tokens, speech_maximum_seconds=speech_maximum_seconds,
            include_generated_descriptions=include_generated_descriptions, dedupe_previous=dedupe_previous,
            passage_prefix=passage_prefix, generated_semantic_only=generated_semantic_only)
        span_words = {span["span_id"]: set(span["word_ids"]) for s in before for span in s["speech"]["spans"]}
        ad_words = {word_id for s in segments for word_id in s["speech"].get("promotional_word_ids", [])}
        for chunk in original:
            if chunk["source_type"] != "speech":
                continue
            ids = set().union(*(span_words.get(sid, set()) for sid in chunk["source_ids"]))
            excluded = ids & ad_words
            if excluded:
                ad_review.append({**chunk, "review_status": "removed" if ids <= ad_words else "partially_filtered",
                                  "excluded_word_ids": sorted(excluded),
                                  "retained_word_ids": sorted(ids - ad_words)})
        write_json(directory / "ad_chunk_review.json", {
            "before_chunks": len(original), "after_chunks": len(chunks),
            "chunk_count_delta": len(chunks) - len(original),
            "removed_chunks": sum(c["review_status"] == "removed" for c in ad_review),
            "partially_filtered_chunks": sum(c["review_status"] == "partially_filtered" for c in ad_review),
            "chunks": ad_review})
    # Counts are checked above: no silent encoder truncation.
    vectors = model.encode(
        [chunk["embedding_text"] for chunk in chunks],
        batch_size=batch_size, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    if (vectors.ndim != 2 or len(vectors) != len(chunks)
            or not np.isfinite(vectors).all()
            or not np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)):
        raise ValueError("Embedding không hợp lệ hoặc chưa được chuẩn hóa.")
    np.save(directory / "embeddings.npy", vectors, allow_pickle=False)
    write_json(directory / "chunk_ids.json", [chunk["chunk_id"] for chunk in chunks])
    (directory / "chunks.jsonl").write_text(
        "\n".join(json.dumps(chunk, ensure_ascii=False, allow_nan=False)
                  for chunk in chunks) + "\n", encoding="utf-8",
    )
    build_lexical_index(directory / "lexical.sqlite3", chunks)
    # Notebook 06 needs only this stage's output: ship its evidence assets with it.
    copy_stage_assets(stage04_dir, directory / "lecture", parent)
    write_json(directory / "lecture/bundle_manifest.json", parent)
    auto_model = model._first_module().auto_model
    resolved_revision = getattr(auto_model.config, "_commit_hash", None)
    parameters = {
        "embedding_model": embedding_model, "revision": revision,
        "resolved_revision": resolved_revision,
        "query_prefix": query_prefix, "passage_prefix": passage_prefix,
        "normalize_embeddings": True, "model_limit": model_limit,
        "visual_description_policy": "structured_json_only" if include_generated_descriptions
                                     else "human_verified_only",
        "generated_semantic_only": generated_semantic_only,
        "speech_chunk_tokens": speech_chunk_tokens,
        "speech_maximum_seconds": speech_maximum_seconds,
        "chunk_tokens": chunk_tokens, "overlap_tokens": overlap_tokens,
        "dimensions": int(vectors.shape[1]),
        "packages": package_versions(["sentence-transformers", "transformers", "numpy"]),
    }
    manifest = finish_stage(
        directory, 5, video_id, parent["source"], parameters, parents=[parent],
        extra={
            "chunk_count": len(chunks), "index_status": "ready",
            "excluded_visual_description_segment_ids": [
                segment["segment_id"] for segment in segments
                if segment["visual"].get("description_status") != "json"
                or (not include_generated_descriptions and segment["visual"].get(
                    "description_verification") != "human_verified")
            ],
        },
    )
    return directory, manifest, chunks


## 3. Cấu hình

Đường dẫn None sẽ được tìm trong Input. Nếu có nhiều phiên bản, điền đường dẫn rõ ràng để tránh chọn nhầm.


In [ ]:
from pathlib import Path

VIDEO_ID = 'L25_V001'
OUTPUT_ROOT = Path('/kaggle/working/lecture_rag') / VIDEO_ID

STAGE_04_DIR = None
EMBEDDING_MODEL = 'intfloat/multilingual-e5-small'
EMBEDDING_REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
CHUNK_TOKENS = 350
OVERLAP_TOKENS = 40
BATCH_SIZE = 32
EMBEDDING_DEVICE = None  # Tự chọn; hoặc 'cpu'/'cuda'
SPEECH_CHUNK_TOKENS = 180
SPEECH_MAXIMUM_SECONDS = 50.0
INCLUDE_GENERATED_DESCRIPTIONS = False  # Chỉ bật cho ablation; JSON hợp lệ vẫn có thể sai nội dung
GENERATED_SEMANTIC_ONLY = False  # Bật cùng mô tả tự động để chỉ dùng nhánh semantic


## 4. Thực thi


In [ ]:
stage04_dir, _ = find_stage(4, STAGE_04_DIR, VIDEO_ID)
output_dir, manifest, chunks = run_indexing(
    stage04_dir, OUTPUT_ROOT, VIDEO_ID,
    embedding_model=EMBEDDING_MODEL, revision=EMBEDDING_REVISION,
    chunk_tokens=CHUNK_TOKENS, overlap_tokens=OVERLAP_TOKENS,
    batch_size=BATCH_SIZE, device=EMBEDDING_DEVICE,
    speech_chunk_tokens=SPEECH_CHUNK_TOKENS, speech_maximum_seconds=SPEECH_MAXIMUM_SECONDS,
    include_generated_descriptions=INCLUDE_GENERATED_DESCRIPTIONS,
    generated_semantic_only=GENERATED_SEMANTIC_ONLY,
)
print('Số chunk:', len(chunks), '| Chiều vector:', manifest['parameters']['dimensions'])


## 5. Kiểm tra / sử dụng kết quả


In [ ]:
from collections import Counter
print('Nguồn:', dict(Counter(chunk['source_type'] for chunk in chunks)))
print('Token tối đa:', max(chunk['encoded_tokens'] for chunk in chunks))
for chunk in chunks[:5]:
    print(chunk['chunk_id'], chunk['start_sec'], chunk['end_sec'])
    print(chunk['text'][:450], '\n')


## Tiếp theo

Save Version; notebook **06** chỉ cần Output này. Đổi embedding
model cần tạo lại chỉ mục. Đây là baseline tìm hình qua OCR/mô tả; chưa dùng
image embeddings. Kết quả vẫn trả lại hình gốc để đối chiếu.
